# PS2 End-to-End Claim Pipeline
This notebook now covers the full project path: dataset inventory, OCR extraction, image signal loading, claim normalization, package-specific output drafting, and NHA model calls for claim correlation.

Run the cells from top to bottom.
Cell 2 defines the reusable helpers.
Cell 3 runs OCR, builds claim tables, validates one claim per package, and exports the working outputs.

In [5]:
from pathlib import Path
import base64
import hashlib
import json
import os
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image

def _resolve_claims_dir(root: Path) -> Path:
    candidates = [
        root / "dataset" / "Claims",
        root / "841403b5-bb4b-4e2a-a73f-570c1b1af8fb" / "Claims",
        root / "Claims",
    ]
    for cand in candidates:
        if cand.exists() and cand.is_dir():
            return cand
    for p in root.rglob("Claims"):
        if p.is_dir():
            return p
    return root / "dataset" / "Claims"

ROOT = Path.cwd()
CLAIMS_DIR = _resolve_claims_dir(ROOT)
OUT_DIR = ROOT / "outputs"
OCR_TEXT_DIR = OUT_DIR / "ocr_texts"
MODEL_OUT_DIR = OUT_DIR / "model_outputs"
for folder in (OUT_DIR, OCR_TEXT_DIR, MODEL_OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Resolved CLAIMS_DIR:", CLAIMS_DIR)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
PDF_EXTS = {".pdf"}

PACKAGE_RULES = {
    "MC011A": {
        "package_name": "PTCA",
        "required_fields": [
            "claim_no", "pre_procedure_image_type", "post_procedure_image_type", "pre_image_description",
            "post_image_description", "stent_visualization", "pre_intra_report_consistency",
            "pre_intra_reason", "pre_inter_report_correlation", "pre_inter_reason",
            "post_inter_report_correlation", "post_inter_reason", "claim_package_alignment",
            "claim_package_reason", "stg_alignment_outcome", "stg_reason", "admission_date",
            "pre_procedure_investigation_date", "post_procedure_investigation_date", "discharge_date",
            "pre_one_month_old_flag", "post_timeline_flag", "quality", "summary_pdf",
        ],
    },
    "MG029A": {
        "package_name": "COPD",
        "required_fields": [
            "claim_no", "post_procedure_image_type", "lung_fields", "cp_angles", "hilum",
            "midline_shift", "cardiac_size", "post_intra_report_consistency", "post_intra_reason",
            "post_inter_report_correlation", "post_inter_reason", "claim_package_alignment",
            "claim_package_reason", "stg_alignment_outcome", "stg_reason", "admission_date",
            "pre_procedure_investigation_date", "post_procedure_investigation_date", "discharge_date",
            "post_timeline_flag", "quality", "summary_pdf",
        ],
    },
    "SG039": {
        "package_name": "Laparoscopic Cholecystectomy",
        "required_fields": [
            "claim_no", "pre_procedure_image_type", "liver", "gall_bladder", "spleen", "kidneys",
            "urinary_bladder", "prostate_or_uterus", "peritoneal_fluid", "pericholecystic_fluid",
            "intra_report_consistency", "intra_reason", "inter_report_correlation", "inter_reason",
            "claim_package_alignment", "claim_package_reason", "stg_alignment_outcome", "stg_reason",
            "admission_date", "pre_procedure_investigation_date", "post_procedure_investigation_date",
            "discharge_date", "pre_one_month_old_flag", "quality", "summary_pdf",
        ],
    },
    "SU007A": {
        "package_name": "PCNL",
        "required_fields": [
            "claim_no", "pre_procedure_image_type", "post_procedure_image_type", "pcs", "ureter",
            "urinary_bladder", "stone_visualization", "radio_opaque_shadow", "dj_stent_or_pcn",
            "pre_intra_report_consistency", "pre_intra_reason", "pre_inter_report_correlation",
            "pre_inter_reason", "post_intra_report_consistency", "post_intra_reason",
            "post_inter_report_correlation", "post_inter_reason", "claim_package_alignment",
            "claim_package_reason", "stg_alignment_outcome", "stg_reason", "admission_date",
            "pre_procedure_investigation_date", "post_procedure_investigation_date", "discharge_date",
            "pre_one_month_old_flag", "post_timeline_flag", "quality", "summary_pdf",
        ],
    },
}

def _safe_rel(path_obj: Path, base: Path) -> str:
    try:
        return str(path_obj.relative_to(base))
    except Exception:
        return str(path_obj)

def _extract_claim_key(path_str: str) -> Tuple[str, str]:
    parts = Path(str(path_str)).parts
    if "Claims" in parts:
        idx = parts.index("Claims")
        package_code = parts[idx + 1] if len(parts) > idx + 1 else "UNKNOWN_PKG"
        claim_id = parts[idx + 2] if len(parts) > idx + 2 else "UNKNOWN_CLAIM"
        return package_code, claim_id
    return "UNKNOWN_PKG", "UNKNOWN_CLAIM"

def _to_bool_series(df: pd.DataFrame, col: str) -> pd.Series:
    if df.empty or col not in df.columns:
        return pd.Series([False] * len(df), index=df.index)
    if df[col].dtype == bool:
        return df[col].fillna(False)
    return df[col].astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])

def _normalize_text(value: Any, default: str = "Not assessed / Not seen") -> str:
    text = "" if value is None else str(value).strip()
    return text if text else default

def load_existing_image_analysis() -> pd.DataFrame:
    img_csv = OUT_DIR / "image_analysis_results.csv"
    if not img_csv.exists():
        return pd.DataFrame()
    df = pd.read_csv(img_csv)
    if "package_code" not in df.columns or "claim_id" not in df.columns:
        keys = df["file_path"].apply(lambda p: pd.Series(_extract_claim_key(p), index=["package_code", "claim_id"]))
        df = pd.concat([df, keys], axis=1)
    return df

def load_existing_ocr_results() -> pd.DataFrame:
    ocr_csv = OUT_DIR / "ocr_batch_results.csv"
    if not ocr_csv.exists():
        return pd.DataFrame()
    df = pd.read_csv(ocr_csv)
    if "package_code" not in df.columns or "claim_id" not in df.columns:
        keys = df["file_path"].apply(lambda p: pd.Series(_extract_claim_key(p), index=["package_code", "claim_id"]))
        df = pd.concat([df, keys], axis=1)
    return df

def get_easyocr_reader(project_root: Path, languages: Optional[List[str]] = None, gpu: bool = False):
    import easyocr
    model_dir = project_root / "assets" / "easyocr_models"
    user_net_dir = project_root / "assets" / "easyocr_user_network"
    model_dir.mkdir(parents=True, exist_ok=True)
    user_net_dir.mkdir(parents=True, exist_ok=True)
    os.environ["EASYOCR_MODULE_PATH"] = str(model_dir)
    os.environ["MODULE_PATH"] = str(model_dir)
    return easyocr.Reader(
        languages or ["en"],
        gpu=gpu,
        model_storage_directory=str(model_dir),
        user_network_directory=str(user_net_dir),
        download_enabled=False,
    )

def ocr_image_with_easyocr(pil_image: Image.Image, reader) -> str:
    results = reader.readtext(np.array(pil_image), detail=0, paragraph=True)
    return "\n".join([item for item in results if isinstance(item, str)]).strip()

def extract_text_from_pdf(pdf_path: Path, max_pages: int = 3, use_gpu: bool = False):
    text = ""
    method = "none"
    error_note = ""
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(pdf_path))
        chunks = [(page.extract_text() or "").strip() for page in reader.pages[:max_pages]]
        direct_text = "\n".join([chunk for chunk in chunks if chunk]).strip()
        if direct_text:
            text = direct_text
            method = "pypdf"
    except Exception as exc:
        error_note = f"pypdf_error={repr(exc)}"
    if len(text) < 120:
        try:
            from pdf2image import convert_from_path
            reader = get_easyocr_reader(ROOT, languages=["en"], gpu=use_gpu)
            images = convert_from_path(str(pdf_path), first_page=1, last_page=max_pages)
            ocr_text = "\n".join([ocr_image_with_easyocr(img, reader) for img in images]).strip()
            if len(ocr_text) > len(text):
                text = ocr_text
                method = "pdf2image+easyocr"
        except Exception as exc:
            extra = f"ocr_error={repr(exc)}"
            error_note = f"{error_note}; {extra}" if error_note else extra
    return text, method, error_note

def run_ocr_batch(force: bool = False, max_pages: int = 3, limit: Optional[int] = None) -> pd.DataFrame:
    ocr_csv = OUT_DIR / "ocr_batch_results.csv"
    if ocr_csv.exists() and not force:
        return pd.read_csv(ocr_csv)
    pdf_files = sorted([p for p in CLAIMS_DIR.rglob("*") if p.is_file() and p.suffix.lower() in PDF_EXTS])
    if limit is not None:
        pdf_files = pdf_files[:limit]
    rows = []
    for idx, pdf_path in enumerate(pdf_files, start=1):
        package_code, claim_id = _extract_claim_key(str(pdf_path))
        try:
            text, method, error_note = extract_text_from_pdf(pdf_path, max_pages=max_pages, use_gpu=False)
            short_hash = hashlib.md5(str(pdf_path).encode("utf-8")).hexdigest()[:10]
            txt_path = OCR_TEXT_DIR / f"{pdf_path.stem}_{short_hash}.txt"
            txt_path.write_text(text, encoding="utf-8", errors="ignore")
            rows.append({
                "file_path": str(pdf_path),
                "relative_path": _safe_rel(pdf_path, CLAIMS_DIR),
                "package_code": package_code,
                "claim_id": claim_id,
                "file_name": pdf_path.name,
                "ocr_method": method,
                "char_count": len(text),
                "low_quality": len(text) < 200,
                "text_file": str(txt_path),
                "text_preview": text[:1000],
                "error": error_note,
            })
            print(f"[{idx:03d}/{len(pdf_files)}] OK | {pdf_path.name} | {method} | chars={len(text)}")
        except Exception as exc:
            rows.append({
                "file_path": str(pdf_path),
                "relative_path": _safe_rel(pdf_path, CLAIMS_DIR),
                "package_code": package_code,
                "claim_id": claim_id,
                "file_name": pdf_path.name,
                "ocr_method": "error",
                "char_count": 0,
                "low_quality": True,
                "text_file": "",
                "text_preview": "",
                "error": repr(exc),
            })
            print(f"[{idx:03d}/{len(pdf_files)}] ERROR | {pdf_path.name} | {repr(exc)}")
    df = pd.DataFrame(rows)
    df.to_csv(ocr_csv, index=False, encoding="utf-8")
    return df

def build_claim_table(df_ocr: Optional[pd.DataFrame] = None, df_img: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    df_ocr = load_existing_ocr_results() if df_ocr is None else df_ocr
    df_img = load_existing_image_analysis() if df_img is None else df_img
    if df_img.empty:
        raise RuntimeError("image_analysis_results.csv is missing. Generate image outputs first.")
    if df_ocr.empty:
        df_ocr = pd.DataFrame(columns=["file_path", "package_code", "claim_id", "char_count", "low_quality", "error"])
    if "package_code" not in df_ocr.columns:
        keys = df_ocr["file_path"].apply(lambda p: pd.Series(_extract_claim_key(p), index=["package_code", "claim_id"]))
        df_ocr = pd.concat([df_ocr, keys], axis=1)
    if "package_code" not in df_img.columns:
        keys = df_img["file_path"].apply(lambda p: pd.Series(_extract_claim_key(p), index=["package_code", "claim_id"]))
        df_img = pd.concat([df_img, keys], axis=1)
    claim_keys = sorted(set(zip(df_ocr["package_code"], df_ocr["claim_id"])) | set(zip(df_img["package_code"], df_img["claim_id"])))
    rows = []
    for package_code, claim_id in claim_keys:
        ocr_slice = df_ocr[(df_ocr["package_code"] == package_code) & (df_ocr["claim_id"] == claim_id)]
        img_slice = df_img[(df_img["package_code"] == package_code) & (df_img["claim_id"] == claim_id)]
        rows.append({
            "package_code": package_code,
            "claim_id": claim_id,
            "ocr_pdf_docs": int(len(ocr_slice)),
            "ocr_nonempty_docs": int((pd.to_numeric(ocr_slice.get("char_count", pd.Series(dtype=int)), errors="coerce").fillna(0) > 0).sum()) if not ocr_slice.empty else 0,
            "ocr_low_quality_docs": int(_to_bool_series(ocr_slice, "low_quality").sum()) if not ocr_slice.empty else 0,
            "ocr_error_docs": int((ocr_slice.get("ocr_method", pd.Series(dtype=str)) == "error").sum()) if not ocr_slice.empty else 0,
            "ocr_total_chars": int(pd.to_numeric(ocr_slice.get("char_count", pd.Series(dtype=int)), errors="coerce").fillna(0).sum()) if not ocr_slice.empty else 0,
            "ocr_avg_chars": float(pd.to_numeric(ocr_slice.get("char_count", pd.Series(dtype=float)), errors="coerce").fillna(0).mean()) if not ocr_slice.empty else 0.0,
            "ocr_max_chars": int(pd.to_numeric(ocr_slice.get("char_count", pd.Series(dtype=int)), errors="coerce").fillna(0).max()) if not ocr_slice.empty else 0,
            "image_docs": int(len(img_slice)),
            "image_low_light_docs": int(_to_bool_series(img_slice, "low_light").sum()) if not img_slice.empty else 0,
            "image_low_contrast_docs": int(_to_bool_series(img_slice, "low_contrast").sum()) if not img_slice.empty else 0,
            "image_tiny_docs": int(_to_bool_series(img_slice, "tiny_image").sum()) if not img_slice.empty else 0,
            "image_error_docs": int((img_slice.get("error", pd.Series(dtype=str)).astype(str).str.len() > 0).sum()) if not img_slice.empty else 0,
            "doc_total": int(len(ocr_slice) + len(img_slice)),
            "sample_ocr_preview": "\n\n".join([str(v) for v in ocr_slice.get("text_preview", pd.Series(dtype=str)).head(2).tolist() if str(v).strip()]),
        })
    return pd.DataFrame(rows).sort_values(["package_code", "claim_id"]).reset_index(drop=True) if rows else pd.DataFrame()

def _load_nha_credentials(root_dir: Path) -> Tuple[str, str, str]:
    env_id = os.getenv("NHA_CLIENT_ID", "").strip()
    env_secret = os.getenv("NHA_CLIENT_SECRET", "").strip()
    if env_id and env_secret:
        return env_id, env_secret, "environment"
    cred_path = root_dir / "client-credentials.json"
    if cred_path.exists():
        with open(cred_path, "r", encoding="utf-8") as handle:
            cred = json.load(handle)
        file_id = str(cred.get("clientId", "")).strip()
        file_secret = str(cred.get("clientSecret", "")).strip()
        if file_id and file_secret:
            return file_id, file_secret, str(cred_path)
    raise RuntimeError("NHA credentials not found. Set NHA_CLIENT_ID and NHA_CLIENT_SECRET or create client-credentials.json.")

def load_nha_client():
    from nha_client import NHAclient
    client_id, client_secret, cred_source = _load_nha_credentials(ROOT)
    print("NHA credentials source:", cred_source)
    return NHAclient(client_id, client_secret)

def image_to_data_url(image_path: Path) -> str:
    suffix = image_path.suffix.lower()
    if suffix in {".jpg", ".jpeg"}:
        mime = "image/jpeg"
    elif suffix == ".png":
        mime = "image/png"
    elif suffix == ".bmp":
        mime = "image/bmp"
    elif suffix in {".tif", ".tiff"}:
        mime = "image/tiff"
    elif suffix == ".webp":
        mime = "image/webp"
    else:
        mime = "image/jpeg"
    with open(image_path, "rb") as handle:
        image_base64 = base64.b64encode(handle.read()).decode("utf-8")
    return f"data:{mime};base64,{image_base64}"

def extract_text_from_response(response: Any) -> str:
    try:
        if isinstance(response, dict):
            choices = response.get("choices", [])
            if choices:
                content = choices[0].get("message", {}).get("content", "")
                if isinstance(content, str):
                    return content.strip()
                if isinstance(content, list):
                    parts = []
                    for item in content:
                        if isinstance(item, dict) and item.get("type") == "text":
                            parts.append(str(item.get("text", "")))
                    return "\n".join([part for part in parts if part]).strip()
            return json.dumps(response, ensure_ascii=False)
        return str(response).strip()
    except Exception:
        return str(response).strip()

def select_representative_images(df_img: pd.DataFrame, package_code: str, claim_id: str, max_images: int = 2) -> List[Path]:
    if df_img.empty:
        return []
    subset = df_img[(df_img["package_code"] == package_code) & (df_img["claim_id"] == claim_id)]
    if subset.empty:
        return []
    subset = subset.sort_values(["file_name"] if "file_name" in subset.columns else ["file_path"])
    return [Path(path) for path in subset["file_path"].head(max_images).tolist()]

def package_prompt(package_code: str, claim_row: pd.Series, evidence_text: str) -> str:
    rule = PACKAGE_RULES.get(package_code, {})
    package_name = rule.get("package_name", package_code)
    required_fields = rule.get("required_fields", [])
    fields_text = "\n".join([f"- {field}" for field in required_fields])
    return (
        f"You are reviewing Problem Statement 2 for package {package_code} ({package_name}).\n"
        f"Use observational language only. Do not make approval, rejection, or diagnostic decisions.\n"
        f"Return a JSON object with these fields:\n{fields_text}\n\n"
        f"Claim evidence:\n{evidence_text}\n\n"
        f"Claim metadata:\npackage_code={claim_row.get('package_code', '')}\nclaim_id={claim_row.get('claim_id', '')}\n"
        f"Observed quality flags: low_light={claim_row.get('image_low_light_docs', 0)}, low_contrast={claim_row.get('image_low_contrast_docs', 0)}, ocr_docs={claim_row.get('ocr_pdf_docs', 0)}\n"
        f"Fill missing fields with 'Not assessed / Not seen'."
    )

def run_nha_claim_inference(nc, image_paths: List[Path], prompt: str, model_name: str) -> str:
    if not image_paths:
        raise RuntimeError("No image paths were provided for model inference.")
    content = [{"type": "image_url", "image_url": {"url": image_to_data_url(path)}} for path in image_paths]
    content.append({"type": "text", "text": prompt})
    response = nc.completion(
        model=model_name,
        messages=[{"role": "user", "content": content}],
        metadata={"problem_statement": 2},
    )
    return extract_text_from_response(response)

def build_structured_placeholder(package_code: str, claim_row: pd.Series) -> Dict[str, Any]:
    fields = PACKAGE_RULES.get(package_code, {}).get("required_fields", [])
    payload = {field: "Not assessed / Not seen" for field in fields}
    payload["claim_no"] = claim_row.get("claim_id", "")
    return payload

def build_submission_row(package_code: str, claim_row: pd.Series, model_text: str, image_paths: List[Path]) -> Dict[str, Any]:
    schema = build_structured_placeholder(package_code, claim_row)
    schema["summary_pdf"] = (
        f"Claim {claim_row.get('claim_id', '')} in package {package_code} has {int(claim_row.get('ocr_pdf_docs', 0))} PDF reports and {int(claim_row.get('image_docs', 0))} images. "
        f"Model text was captured for correlation and summary generation."
    )
    return {
        "package_code": package_code,
        "claim_id": claim_row.get("claim_id", ""),
        "model_name": os.getenv("NHA_MODEL", "Ministral 8B"),
        "model_used": bool(model_text.strip()),
        "representative_images": "|".join([str(path) for path in image_paths]),
        "structured_fields_json": json.dumps(schema, ensure_ascii=False),
        "summary_text": schema["summary_pdf"],
        "model_text": model_text,
    }

def export_outputs(df_claims: pd.DataFrame, model_df: pd.DataFrame) -> None:
    claim_out = OUT_DIR / "claims_aggregated.csv"
    df_claims.to_csv(claim_out, index=False, encoding="utf-8")
    final_out = OUT_DIR / "final_claim_outputs.csv"
    model_df.to_csv(final_out, index=False, encoding="utf-8")
    summary_path = OUT_DIR / "final_claim_summaries.md"
    with open(summary_path, "w", encoding="utf-8") as handle:
        for _, row in model_df.iterrows():
            handle.write(f"## {row['package_code']} / {row['claim_id']}\n\n")
            handle.write(row["summary_text"] + "\n\n")
    print("Saved:", claim_out)
    print("Saved:", final_out)
    print("Saved:", summary_path)

Resolved CLAIMS_DIR: c:\AB-PMJAY-Hackathon\dataset\Claims


In [6]:
import importlib.util

RUN_OCR_BATCH = False
RUN_FULL_MODEL_BATCH = True

print("=== Project Run Start ===")
print("Workspace:", ROOT)
print("Claims dir exists:", CLAIMS_DIR.exists())

df_images = load_existing_image_analysis()
if df_images.empty:
    raise RuntimeError("outputs/image_analysis_results.csv is missing. Build image outputs before running this cell.")
print("Image rows loaded:", len(df_images))

if RUN_OCR_BATCH:
    df_ocr = run_ocr_batch(force=True, max_pages=3)
else:
    df_ocr = load_existing_ocr_results()
print("OCR rows loaded:", len(df_ocr))

df_claims = build_claim_table(df_ocr=df_ocr, df_img=df_images)
claims_out = OUT_DIR / "claims_aggregated.csv"
df_claims.to_csv(claims_out, index=False, encoding="utf-8")
print("Claim rows:", len(df_claims))
print("Claims CSV saved:", claims_out)

if df_claims.empty:
    raise RuntimeError("No claim rows were produced after aggregation.")

sample_claims = df_claims.sort_values(["package_code", "claim_id"]).groupby("package_code", as_index=False).head(1).reset_index(drop=True)
print("Sample claims for validation:")
print(sample_claims[["package_code", "claim_id", "ocr_pdf_docs", "image_docs", "doc_total"]].to_string(index=False))

model_rows = []
model_name = os.getenv("NHA_MODEL", "Ministral 8B")
nha_module_available = importlib.util.find_spec("nha_client") is not None
nha_ready = False
nc = None

if RUN_FULL_MODEL_BATCH and not nha_module_available:
    print("nha_client package is not available in this kernel. Exporting placeholder structured rows.")
elif RUN_FULL_MODEL_BATCH and nha_module_available:
    try:
        nc = load_nha_client()
        nha_ready = True
    except RuntimeError as exc:
        print("NHA credentials are missing. Exporting placeholder rows instead of live model calls.")
        print("Detail:", str(exc))
        print("Fix: set NHA_CLIENT_ID and NHA_CLIENT_SECRET env vars, or place client-credentials.json in notebook root.")

if RUN_FULL_MODEL_BATCH and nha_ready:
    for idx, claim_row in df_claims.iterrows():
        package_code = str(claim_row["package_code"])
        claim_id = str(claim_row["claim_id"])
        claim_images = select_representative_images(df_images, package_code, claim_id, max_images=2)
        evidence_text = (
            f"OCR docs: {int(claim_row.get('ocr_pdf_docs', 0))}\n"
            f"Non-empty OCR docs: {int(claim_row.get('ocr_nonempty_docs', 0))}\n"
            f"Image docs: {int(claim_row.get('image_docs', 0))}\n"
            f"OCR preview:\n{claim_row.get('sample_ocr_preview', '')}"
        )
        prompt = package_prompt(package_code, claim_row, evidence_text)
        try:
            model_text = run_nha_claim_inference(nc, claim_images, prompt, model_name)
        except Exception as exc:
            model_text = f"MODEL_ERROR: {repr(exc)}"
        model_rows.append(build_submission_row(package_code, claim_row, model_text, claim_images))
        print(f"[{idx + 1:03d}/{len(df_claims)}] {package_code} | {claim_id} | images={len(claim_images)}")
else:
    if not RUN_FULL_MODEL_BATCH:
        print("Model batch disabled. Exporting placeholder structured rows.")
    for _, claim_row in df_claims.iterrows():
        package_code = str(claim_row["package_code"])
        claim_id = str(claim_row["claim_id"])
        claim_images = select_representative_images(df_images, package_code, claim_id, max_images=2)
        model_rows.append(build_submission_row(package_code, claim_row, "", claim_images))

df_model = pd.DataFrame(model_rows)
model_out = OUT_DIR / "final_claim_outputs.csv"
df_model.to_csv(model_out, index=False, encoding="utf-8")

summary_path = OUT_DIR / "final_claim_summaries.md"
with open(summary_path, "w", encoding="utf-8") as handle:
    for _, row in df_model.iterrows():
        handle.write(f"## {row['package_code']} / {row['claim_id']}\n\n")
        handle.write(row["summary_text"] + "\n\n")

print("Model rows saved:", model_out)
print("Summary saved:", summary_path)
print("=== Project Run Complete ===")
print(df_model.head().to_string(index=False))

=== Project Run Start ===
Workspace: c:\AB-PMJAY-Hackathon
Claims dir exists: True
Image rows loaded: 37
OCR rows loaded: 55
Claim rows: 40
Claims CSV saved: c:\AB-PMJAY-Hackathon\outputs\claims_aggregated.csv
Sample claims for validation:
package_code                          claim_id  ocr_pdf_docs  image_docs  doc_total
      MC011A    BOCW_GJ_R3_2026040310046613_ER             2           2          4
      MG029A  BOCW_UP_2025_R2_2026032410041507             1           1          2
       SG039  BOCW_UP_2025_R2_2026032310064837             0           2          2
      SU007A MMJAA_UP_2026_R3_2026033010053802             2           0          2
nha_client package is not available in this kernel. Exporting placeholder structured rows.
Model rows saved: c:\AB-PMJAY-Hackathon\outputs\final_claim_outputs.csv
Summary saved: c:\AB-PMJAY-Hackathon\outputs\final_claim_summaries.md
=== Project Run Complete ===
package_code                            claim_id   model_name  model_used     